In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile

# 2. Define project paths
project_path = '/content/drive/MyDrive/face_recognition_project'

# Point to your zip file - CORRECTED to use Original Images.zip
zip_file_path = os.path.join(project_path, 'Original Images.zip')
extracted_images_path = '/content/images'

# 3. Unzip to Colab's local temporary storage
print("Unzipping images to local Colab storage...")

# Check if the zip file exists before attempting to open it
if not os.path.exists(zip_file_path):
    print(f"Error: The zip file was not found at '{zip_file_path}'")
    print("Please ensure the file exists in your Google Drive and the path is correct.")
else:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_images_path)

    print("Done! Images unzipped successfully to /content/images")

In [ ]:
import pandas as pd

dataset_path = '/content/drive/MyDrive/face_recognition_project/Dataset.csv'

# Load the dataset
df_dataset = pd.read_csv(dataset_path)

# Display the first 5 rows of the dataset
print("First 5 rows of the dataset:")
display(df_dataset.head())

In [ ]:
import os

project_path = '/content/drive/MyDrive/face_recognition_project'

print(f"Contents of '{project_path}':")
if os.path.exists(project_path):
    contents = os.listdir(project_path)
    if contents:
        for item in contents:
            print(item)
    else:
        print(f"The directory '{project_path}' is empty.")
else:
    print(f"Error: Project path '{project_path}' does not exist. Please check your Google Drive mount and path.")

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive & Install Dependencies
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Install required libraries
# Note: dlib compilation can take 3-5 minutes on Colab
!pip install face-recognition opencv-python dlib pandas tqdm -q

# Verify installation
import face_recognition
import cv2
import pandas as pd
from tqdm import tqdm

print("All dependencies installed successfully")
print(f"face_recognition version check: OK")

In [ ]:
# ============================================================
# CELL 2: Stratified 80/20 Train/Test Split
# ============================================================
# Splits Dataset.csv into train_labels.csv (80%) and test_labels.csv (20%),
# stratified by label so each person is proportionally represented in both sets.
# Saved to Drive so Cell 3 (encoding) and Cell 6 (evaluation) can reuse them
# across sessions without re-splitting.

import os
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_DIR = '/content/drive/MyDrive/face_recognition_project'
CSV_PATH = os.path.join(PROJECT_DIR, 'Dataset.csv')
TRAIN_CSV_PATH = os.path.join(PROJECT_DIR, 'train_labels.csv')
TEST_CSV_PATH = os.path.join(PROJECT_DIR, 'test_labels.csv')

RANDOM_SEED = 42

df = pd.read_csv(CSV_PATH)
print(f"Total images: {len(df)}")
print(f"Total unique people: {df['label'].nunique()}")

# --- Handle classes with too few samples to stratify ---
# sklearn's stratify requires at least 2 samples per class (so each side of
# the split gets at least 1). Classes with only 1 image can't be split
# meaningfully — they're routed entirely into the training set instead,
# since a lone example can't be both trained on and tested against.
label_counts = df['label'].value_counts()
single_sample_labels = label_counts[label_counts < 2].index.tolist()

if single_sample_labels:
    print(f"\n  {len(single_sample_labels)} people have only 1 image — "
          f"they'll be placed entirely in the training set (can't be stratified):")
    for lbl in single_sample_labels[:10]:
        print(f"   - {lbl}")
    if len(single_sample_labels) > 10:
        print(f"   ... and {len(single_sample_labels) - 10} more")

splittable_df = df[~df['label'].isin(single_sample_labels)]
forced_train_df = df[df['label'].isin(single_sample_labels)]

# --- Stratified split on the splittable portion ---
train_split, test_split = train_test_split(
    splittable_df,
    test_size=0.2,
    stratify=splittable_df['label'],
    random_state=RANDOM_SEED
)

# Merge forced-train singletons back into the training set
train_df = pd.concat([train_split, forced_train_df], ignore_index=True)
test_df = test_split.reset_index(drop=True)

# --- Save to Drive ---
train_df.to_csv(TRAIN_CSV_PATH, index=False)
test_df.to_csv(TEST_CSV_PATH, index=False)

print("\n" + "="*50)
print("SPLIT COMPLETE")
print("="*50)
print(f"Training set: {len(train_df)} images ({len(train_df)/len(df):.1%})")
print(f"Test set:     {len(test_df)} images ({len(test_df)/len(df):.1%})")
print(f"People in training set: {train_df['label'].nunique()}")
print(f"People in test set:     {test_df['label'].nunique()}")
print(f"\nSaved: {TRAIN_CSV_PATH}")
print(f"Saved: {TEST_CSV_PATH}")

# Sanity check: warn if any person appears in test but never in train
train_people = set(train_df['label'])
test_people = set(test_df['label'])
test_only = test_people - train_people
if test_only:
    print(f"\n  {len(test_only)} people appear in test set but have zero training "
          f"images — they can never be correctly matched:")
    for p in list(test_only)[:10]:
        print(f"   - {p}")

In [ ]:
# ============================================================
# CELL 3: Compute Face Encodings for Training Set Only (FIXED)
# ============================================================

import os
import pickle
import pandas as pd
import face_recognition
from tqdm import tqdm

PROJECT_DIR = '/content/drive/MyDrive/face_recognition_project'
TRAIN_CSV_PATH = os.path.join(PROJECT_DIR, 'train_labels.csv')
ENCODINGS_PATH = os.path.join(PROJECT_DIR, 'encodings.pkl')

# --- Dynamically locate the actual extracted dataset root ---
def find_base_images_dir(start_path='/content/images'):
    for root, dirs, files in os.walk(start_path):
        # Look for a known class folder name from the dataset
        if "Robert Downey Jr" in dirs or "Billie Eilish" in dirs:
            return root
    return start_path

BASE_IMAGES_DIR = find_base_images_dir('/content/images')
print(f" Located dataset image root at: {BASE_IMAGES_DIR}")

df = pd.read_csv(TRAIN_CSV_PATH)
print(f"Loaded {len(df)} TRAINING entries from train_labels.csv")

known_encodings, known_labels, known_ids = [], [], []
skipped_images, error_images = [], []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Encoding training faces"):
    image_id = row['id']
    label = row['label']
    image_path = os.path.join(BASE_IMAGES_DIR, label, image_id)

    if not os.path.exists(image_path):
        error_images.append((image_id, f"File not found at {image_path}"))
        continue

    try:
        image = face_recognition.load_image_file(image_path)
        face_locations = face_recognition.face_locations(image)

        if len(face_locations) == 0:
            skipped_images.append(image_id)
            continue

        face_encodings = face_recognition.face_encodings(image, known_face_locations=[face_locations[0]])

        if len(face_encodings) == 0:
            skipped_images.append(image_id)
            continue

        known_encodings.append(face_encodings[0])
        known_labels.append(label)
        known_ids.append(image_id)

    except Exception as e:
        error_images.append((image_id, str(e)))
        continue

data = {
    "encodings": known_encodings,
    "labels": known_labels,
    "ids": known_ids
}

with open(ENCODINGS_PATH, 'wb') as f:
    pickle.dump(data, f)

print("\n" + "="*50)
print("ENCODING COMPLETE (TRAINING SET)")
print("="*50)
print(f"Total training images:    {len(df)}")
print(f"Successfully encoded:     {len(known_encodings)}")
print(f"Skipped (no face found):  {len(skipped_images)}")
print(f"Errors (missing/corrupt): {len(error_images)}")
print(f"Saved to: {ENCODINGS_PATH}")

 Located dataset image root at: /content/images/Original Images/Original Images
Loaded 2049 TRAINING entries from train_labels.csv


Encoding training faces:  11%|█         | 224/2049 [04:41<38:14,  1.26s/it]


KeyboardInterrupt: 

In [ ]:
import os

print(f"Contents of {extracted_images_path}:")
if os.path.exists(extracted_images_path):
    for item in os.listdir(extracted_images_path):
        print(item)
else:
    print(f"Error: The extracted images path '{extracted_images_path}' does not exist.")

In [ ]:
# ============================================================
# CELL 4: Face Recognition Inference on a Test Image
# ============================================================

import os
import pickle
import numpy as np
import face_recognition
import cv2
from google.colab.patches import cv2_imshow

PROJECT_DIR = '/content/drive/MyDrive/face_recognition_project'
ENCODINGS_PATH = os.path.join(PROJECT_DIR, 'encodings.pkl')

with open(ENCODINGS_PATH, 'rb') as f:
    data = pickle.load(f)

known_encodings = data['encodings']
known_labels = data['labels']

print(f"Loaded {len(known_encodings)} known face encodings")

# Locate base directory dynamically
def find_base_images_dir(start_path='/content/images'):
    for root, dirs, files in os.walk(start_path):
        if "Robert Downey Jr" in dirs:
            return root
    return start_path

BASE_IMAGES_DIR = find_base_images_dir('/content/images')
TEST_IMAGE_PATH = os.path.join(BASE_IMAGES_DIR, 'Robert Downey Jr', 'Robert Downey Jr_87.jpg')

def recognize_faces(image_path, known_encodings, known_labels, tolerance=0.6):
    if not os.path.exists(image_path):
        print(f"Test image not found: {image_path}")
        return

    rgb_image = face_recognition.load_image_file(image_path)
    face_locations = face_recognition.face_locations(rgb_image)
    face_encodings = face_recognition.face_encodings(rgb_image, face_locations)

    if len(face_locations) == 0:
        print("No face detected in test image.")
        return

    bgr_image = cv2.imread(image_path)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        distances = face_recognition.face_distance(known_encodings, face_encoding)
        best_match_index = np.argmin(distances)
        best_distance = distances[best_match_index]

        if best_distance <= tolerance:
            name = known_labels[best_match_index]
            confidence = (1 - best_distance) * 100
            label_text = f"{name} ({confidence:.1f}%)"
            color = (0, 255, 0)
        else:
            label_text = "Unknown"
            color = (0, 0, 255)

        cv2.rectangle(bgr_image, (left, top), (right, bottom), color, 2)
        cv2.rectangle(bgr_image, (left, bottom - 25), (right, bottom), color, cv2.FILLED)
        cv2.putText(bgr_image, label_text, (left + 6, bottom - 6),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)

        print(f"Detected: {label_text}  (distance: {best_distance:.4f})")

    cv2_imshow(bgr_image)

recognize_faces(TEST_IMAGE_PATH, known_encodings, known_labels, tolerance=0.6)

In [ ]:
# ============================================================
# CELL 5: Batch Evaluation on Held-Out Test Set (FIXED)
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd
import face_recognition
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = '/content/drive/MyDrive/face_recognition_project'
ENCODINGS_PATH = os.path.join(PROJECT_DIR, 'encodings.pkl')
TEST_CSV_PATH = os.path.join(PROJECT_DIR, 'test_labels.csv')
TOLERANCE = 0.6

def find_base_images_dir(start_path='/content/images'):
    for root, dirs, files in os.walk(start_path):
        if "Robert Downey Jr" in dirs:
            return root
    return start_path

BASE_TEST_IMAGES_DIR = find_base_images_dir('/content/images')

with open(ENCODINGS_PATH, 'rb') as f:
    data = pickle.load(f)
known_encodings = data['encodings']
known_labels = data['labels']
print(f"Loaded {len(known_encodings)} known encodings")

test_df = pd.read_csv(TEST_CSV_PATH)
test_items = [
    (os.path.join(BASE_TEST_IMAGES_DIR, row['label'], row['id']), row['label'])
    for _, row in test_df.iterrows()
]
print(f"Evaluating on {len(test_items)} held-out TEST images...")

y_true, y_pred, no_face_count, detail_rows = [], [], 0, []

for img_path, true_label in tqdm(test_items, desc="Evaluating"):
    if not os.path.exists(img_path):
        continue
    try:
        image = face_recognition.load_image_file(img_path)
        face_locations = face_recognition.face_locations(image)
        if len(face_locations) == 0:
            no_face_count += 1
            continue

        face_encoding = face_recognition.face_encodings(image, known_face_locations=[face_locations[0]])[0]
        distances = face_recognition.face_distance(known_encodings, face_encoding)
        best_idx = np.argmin(distances)
        best_distance = distances[best_idx]
        predicted_label = known_labels[best_idx] if best_distance <= TOLERANCE else "Unknown"

        y_true.append(true_label)
        y_pred.append(predicted_label)
        detail_rows.append({
            "image": os.path.basename(img_path),
            "true_label": true_label,
            "predicted_label": predicted_label,
            "distance": round(float(best_distance), 4),
            "correct": true_label == predicted_label,
        })
    except Exception as e:
        continue

print("\n" + "="*50)
print("EVALUATION RESULTS (HELD-OUT TEST SET)")
print("="*50)
print(f"Total evaluated:  {len(y_true)}")
print(f"No face detected: {no_face_count}")

if len(y_true) > 0:
    print(f"Overall accuracy: {accuracy_score(y_true, y_pred):.2%}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

In [ ]:
import os

# Change this to inspect the 'Faces' directory
final_images_dir = os.path.join('/content/images', 'Faces')

print(f"Contents of {final_images_dir}:")
if os.path.exists(final_images_dir):
    contents = os.listdir(final_images_dir)
    if contents:
        for item in contents[:20]: # Print first 20 items for brevity
            print(item)
        if len(contents) > 20:
            print(f"... and {len(contents) - 20} more items.")
    else:
        print(f"The directory '{final_images_dir}' is empty.")
else:
    print(f"Error: The directory '{final_images_dir}' does not exist.")

In [ ]:
import os

images_sub_dir = os.path.join('/content/images', 'Original Images')

print(f"Contents of {images_sub_dir}:")
if os.path.exists(images_sub_dir):
    for item in os.listdir(images_sub_dir):
        print(item)
else:
    print(f"Error: The directory '{images_sub_dir}' does not exist.")


In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import numpy as np
import pickle
import os
import face_recognition # Corrected import from face_recognition_project
from PIL import Image

# App Title & Layout
st.set_page_config(page_title="Face Recognition Web App", layout="wide")
st.title("👤 Real-Time Face Recognition Dashboard")

# Updated Path: face_recognition_project (matching your notebook)
project_path = '/content/drive/MyDrive/face_recognition_project'
encodings_file = os.path.join(project_path, 'encodings.pkl')

# Load saved face encodings
@st.cache_resource
def load_encodings():
    if os.path.exists(encodings_file):
        with open(encodings_file, 'rb') as f:
            data = pickle.load(f)
        # Handles both 'labels' and 'names' dictionary keys safely
        names = data.get('labels', data.get('names', []))
        return data['encodings'], names
    return [], []

known_encodings, known_names = load_encodings()

# Sidebar Navigation
st.sidebar.title("Navigation")
app_mode = st.sidebar.radio("Choose Mode", ["Image Upload & Recognition", "Webcam Snapshot Mode", "Model Information"])

# Mode 1: Image Upload
if app_mode == "Image Upload & Recognition":
    st.header("Upload Image for Recognition")
    uploaded_file = st.file_uploader("Choose a photo...", type=["jpg", "jpeg", "png"])

    col1, col2 = st.columns(2)

    if uploaded_file is not None:
        image = Image.open(uploaded_file)
        img_array = np.array(image)
        rgb_image = cv2.cvtColor(img_array, cv2.COLOR_RGBA2RGB) if img_array.shape[2] == 4 else img_array

        with col1:
            st.image(image, caption="Uploaded Image", use_container_width=True)

        if st.button("Recognize Faces"):
            face_locations = face_recognition.face_locations(rgb_image)
            face_encodings = face_recognition.face_encodings(rgb_image, face_locations)

            annotated_image = rgb_image.copy()
            detected_names = []

            for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
                name = "Unknown"
                if len(known_encodings) > 0:
                    distances = face_recognition.face_distance(known_encodings, face_encoding)
                    best_match_idx = np.argmin(distances)
                    if distances[best_match_idx] <= 0.6:
                        name = known_names[best_match_idx]

                detected_names.append(name)

                cv2.rectangle(annotated_image, (left, top), (right, bottom), (0, 255, 0), 2)
                cv2.rectangle(annotated_image, (left, bottom - 25), (right, bottom), (0, 255, 0), cv2.FILLED)
                cv2.putText(annotated_image, name, (left + 6, bottom - 6),
                            cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)

            with col2:
                st.image(annotated_image, caption="Recognition Result", use_container_width=True)
                st.success(f"Detected Faces: {', '.join(detected_names) if detected_names else 'No faces found'}")

# Mode 2: Webcam Snapshot Mode
elif app_mode == "Webcam Snapshot Mode":
    st.header("Capture Photo from Webcam")
    img_file_buffer = st.camera_input("Take a photo")

    if img_file_buffer is not None:
        bytes_data = img_file_buffer.getvalue()
        cv2_img = cv2.imdecode(np.frombuffer(bytes_data, np.uint8), cv2.IMREAD_COLOR)
        rgb_img = cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB)

        face_locations = face_recognition.face_locations(rgb_img)
        face_encodings = face_recognition.face_encodings(rgb_img, face_locations)

        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            name = "Unknown"
            if len(known_encodings) > 0:
                distances = face_recognition.face_distance(known_encodings, face_encoding)
                best_match_idx = np.argmin(distances)
                if distances[best_match_idx] <= 0.6:
                    name = known_names[best_match_idx]

            cv2.rectangle(rgb_img, (left, top), (right, bottom), (0, 255, 0), 2)
            cv2.rectangle(rgb_img, (left, bottom - 25), (right, bottom), (0, 255, 0), cv2.FILLED)
            cv2.putText(rgb_img, name, (left + 6, bottom - 6), cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)

        st.image(rgb_img, caption="Webcam Recognition Result", use_container_width=True)

# Mode 3: Stats
elif app_mode == "Model Information":
    st.header("Model & Dataset Statistics")
    st.write(f"**Total Known Embeddings:** {len(known_encodings)}")
    st.write(f"**Unique Identities:** {len(set(known_names))}")
    if known_names:
        st.write("**Classes/People Enrolled:**")
        st.json(list(set(known_names)))


Overwriting app.py


In [ ]:
!pip install streamlit pyngrok -q

import subprocess
from pyngrok import ngrok

# Kill any existing Streamlit/ngrok processes to free up port 8501
!pkill -f streamlit
!pkill -f ngrok

# Insert your ngrok auth token here
ngrok.set_auth_token("3HyOw95KbEaVhpzIYdQ3TK6byUN_7xaG3KVzJHGVTckzDXev3")

# Launch Streamlit in the background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Create the public tunnel
public_url = ngrok.connect(8501)

print("=" * 60)
print("SUCCESS! Click your Streamlit Web App link below:")
print(f"{public_url}")
print("=" * 60)

SUCCESS! Click your Streamlit Web App link below:
NgrokTunnel: "https://pacifism-breach-detail.ngrok-free.dev" -> "http://localhost:8501"
